In [1]:
import json
import subprocess
import os
import re

# --- Extract variables directly into os.environ ---
workspace = json.loads(subprocess.run(
    ["wb", "workspace", "describe", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

os.environ["GOOGLE_CLOUD_PROJECT"] = workspace["googleProjectId"]

resources = json.loads(subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

# List to collect all matching CDRs
cdr_matches = []
# Updated Pattern: Starts with either 'C' or 'R'
# Groups: (Prefix), (Year), (Quarter), (Release)
cdr_pattern = re.compile(r"^([CR])(\d{4})Q(\d+)R(\d+)$")

# --- Step 3: Extract workspace resources ---
for r in resources:

 # 1. MASTER BUCKET LOGIC (Outer block)
    if r["resourceType"] == "GCS_BUCKET":
        print(f"Found bucket: id={r['id']}, bucketName={r['bucketName']}")
        
        # Inner conditional logic strictly for filtering different bucket IDs
        if "temporary-workspace-bucket" in r["id"]:
            os.environ["WORKSPACE_TEMP_BUCKET"] = f"gs://{r['bucketName']}"
        elif "workspace-bucket" in r["id"]:
            os.environ["WORKSPACE_BUCKET"] = f"gs://{r['bucketName']}"
        elif r["id"] == "aou-tutorial-notebooks":
            os.environ["bucket_aou_tutorial"] = f"gs://{r['bucketName']}"
            os.environ["bucket_id_aou_tutorial"] = r["id"]
            print("-> Assigned aou-tutorial variables")
        elif r["id"].startswith("rw-migration"):
            os.environ["bucket_migrated"] = f"gs://{r['bucketName']}"
            os.environ["bucket_id_migrated"] = r["id"]
            print(f"-> Assigned migration variables (ID: {r['id']})")
            
    # 2. BQ DATASET LOGIC (Handles C... and R...)
    elif r["resourceType"] in ["BQ_DATASET", "BIGQUERY_DATASET"]:
        dataset_id = r["datasetId"]
        match = cdr_pattern.match(dataset_id)
        
        if match:
            prefix, year, quarter, release = match.groups()
            # The sorting key now includes the prefix to keep C and R distinct if needed,
            # but still prioritizes the latest Year/Quarter/Release.
            version_key = (prefix, int(year), int(quarter), int(release))
            full_path = f"{r['projectId']}.{dataset_id}"
            cdr_matches.append((version_key, full_path))

# --- Finalize the WORKSPACE_CDR ---
if cdr_matches:
    # This sorts by Prefix (C before R), then Year, then Quarter, then Release
    cdr_matches.sort(key=lambda x: x[0])
    latest_r = cdr_matches[-1][1]
    
    os.environ["WORKSPACE_CDR"] = latest_r
    print(f"✅ Successfully identified latest dataset: {os.environ['WORKSPACE_CDR']}")
else:
    os.environ["WORKSPACE_CDR"] = ""
    print("⚠️ No C... or R... datasets found.")

# --- Save to .bashrc ---
# --- Expanded tracking array containing all variables ---
vars_to_save = [
    "GOOGLE_CLOUD_PROJECT", 
    "WORKSPACE_BUCKET", 
    "WORKSPACE_TEMP_BUCKET", 
    "WORKSPACE_CDR",
    "bucket_aou_tutorial",     # Path (gs://...)
    "bucket_id_aou_tutorial",  # ID only (aou-tutorial-notebooks)
    "bucket_migrated",         # Path (gs://...)
    "bucket_id_migrated"       # ID only (rw-migration-xxxx)
]

# --- Print validation check ---
print("\nVariables extracted:")
for var in vars_to_save:
    value = os.environ.get(var)
    print(f"{var}: {value if value else 'NOT FOUND'}")


# --- Save to .bashrc ---
bashrc_path = os.path.expanduser("~/.bashrc")

if not os.path.exists(bashrc_path):
    print(f"Creating {bashrc_path}...")
    with open(bashrc_path, "w") as f:
        f.write("# Created by Verily setup script\n")

# Append everything safely to the end of the file
with open(bashrc_path, "a") as f:
    f.write("\n# Verily Workbench variables\n")
    for var in vars_to_save:
        value = os.environ.get(var)
        if value:
            f.write(f'export {var}="{value}"\n')

print(f"\n✅ Saved to {bashrc_path}")

Found bucket: id=dataproc-staging-aou-rw-67cadd48, bucketName=dataproc-staging-wb-chirpy-leek-1262
Found bucket: id=dataproc-temp-aou-rw-67cadd48, bucketName=dataproc-temp-wb-chirpy-leek-1262
Found bucket: id=rw-migration-aou-rw-67cadd48, bucketName=rw-migration-aou-rw-67cadd48
-> Assigned migration variables (ID: rw-migration-aou-rw-67cadd48)
Found bucket: id=vwb-aou-datasets-controlled, bucketName=vwb-aou-datasets-controlled
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-chirpy-leek-1262
WORKSPACE_BUCKET: NOT FOUND
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-67cadd48
bucket_id_migrated: rw-migration-aou-rw-67cadd48

✅ Saved to /home/jupyter/.bashrc


In [2]:
for r in resources:
    print(r["id"], r["resourceType"])
    if r["resourceType"] == "GCS_BUCKET" and "workspace-bucket" in r["id"]:
        os.environ["WORKSPACE_BUCKET"] = f"gs://{r['bucketName']}"
    elif r["resourceType"] == "GCS_BUCKET" and "temporary" in r["id"]:
        os.environ["WORKSPACE_TEMP_BUCKET"] = f"gs://{r['bucketName']}"

C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20260727 DATAPROC_CLUSTER
dataproc-staging-aou-rw-67cadd48 GCS_BUCKET
dataproc-temp-aou-rw-67cadd48 GCS_BUCKET
rw-migration-aou-rw-67cadd48 GCS_BUCKET
vwb-aou-datasets-controlled GCS_BUCKET
